# Week 48, Day 3: Rolling hashes

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w048d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
B, M = 10, 10**9 + 7
s = "31415"

h = [0]
for ch in s:
    h.append((h[-1] * B + int(ch)) % M)
print(h)

def sub(i, j):                       # the hash of s[i:j]
    return (h[j] - h[i] * B ** (j - i)) % M

print(sub(1, 4), sub(2, 5), sub(0, 2))

In [ ]:
class PrefixHash:
    def __init__(self, s, base=131, mod=(1 << 61) - 1):
        self.mod = mod
        self.h = [0] * (len(s) + 1)
        self.pw = [1] * (len(s) + 1)
        for i, ch in enumerate(s):
            self.h[i + 1] = (self.h[i] * base + ord(ch)) % mod
            self.pw[i + 1] = self.pw[i] * base % mod

    def get(self, i, j):
        """The hash of s[i:j]."""
        return (self.h[j] - self.h[i] * self.pw[j - i]) % self.mod

s = "abracadabra"
ph = PrefixHash(s)
print(ph.get(0, 4) == ph.get(7, 11), s[0:4], s[7:11])
print(ph.get(0, 4) == ph.get(1, 5))

In [ ]:
import random

M = 1_000_003                       # deliberately small, to make collisions easy to find
B = 256
def h(s):
    v = 0
    for ch in s:
        v = (v * B + ord(ch)) % M
    return v

rng = random.Random(7)
seen = {}
tries = 0
while True:
    s = "".join(rng.choice("abcdefghij") for _ in range(8))
    tries += 1
    v = h(s)
    if v in seen and seen[v] != s:
        print(f"after {tries} strings: {seen[v]!r} and {s!r} both hash to {v}")
        break
    seen[v] = s

In [ ]:
import random

class PrefixHash:
    def __init__(self, s, base, mod=(1 << 61) - 1):
        self.mod, self.h, self.pw = mod, [0] * (len(s) + 1), [1] * (len(s) + 1)
        for i, ch in enumerate(s):
            self.h[i + 1] = (self.h[i] * base + ord(ch)) % mod
            self.pw[i + 1] = self.pw[i] * base % mod
    def get(self, i, j):
        return (self.h[j] - self.h[i] * self.pw[j - i]) % self.mod

def repeat_of_length(ph, n, length):
    """A start position of a length-`length` substring that occurs twice, or None."""
    seen = set()
    for i in range(n - length + 1):
        v = ph.get(i, i + length)
        if v in seen:
            return i
        seen.add(v)
    return None

def longest_repeat(s):
    ph = PrefixHash(s, random.randrange(256, 1 << 40))
    lo, hi, best = 1, len(s) - 1, ""
    while lo <= hi:
        mid = (lo + hi) // 2
        i = repeat_of_length(ph, len(s), mid)
        if i is None:
            hi = mid - 1
        else:
            best, lo = s[i:i + mid], mid + 1
    return best

print(repr(longest_repeat("banana")), repr(longest_repeat("abcdefg")), repr(longest_repeat("to be or not to be")))

In [ ]:
import random

M = (1 << 61) - 1
B = random.randrange(256, 1 << 40)          # one base, shared by both strings

def window_hashes(s, k):
    """{hash: first start} for every length-k window of s."""
    out, v, high = {}, 0, pow(B, k - 1, M)
    for i, ch in enumerate(s):
        if i >= k:
            v = (v - ord(s[i - k]) * high) % M
        v = (v * B + ord(ch)) % M
        if i >= k - 1:
            out.setdefault(v, i - k + 1)
    return out

a, b = "the cat sat on the mat", "a mat for the cat"
for k in (3, 6, 8):
    ha, hb = window_hashes(a, k), window_hashes(b, k)
    shared = sorted(a[ha[v]:ha[v] + k] for v in ha.keys() & hb.keys())
    print(k, len(ha), "distinct in a; shared:", shared)

> **Key idea.** Prefix hashes `h[i + 1] = h[i]·B + code(s[i])` (mod M) give any substring's hash in O(1): `h[j] − h[i]·B^(j−i)`. Use a large prime modulus such as 2⁶¹ − 1 and a base chosen at random when the program starts, so collisions are vanishingly rare and can't be engineered; check the characters when a mistake would be unacceptable. When a yes/no question about a length is monotonic, binary search the answer: the longest repeated substring and the longest common substring are O(n log n) this way. Sets of window hashes count distinct substrings of a length. Binary searching the longest common prefix with hashes compares substrings alphabetically in O(log n).

### Exercise: Prefix hashes

Write `PrefixHash(s, base=None, mod=(1 << 61) - 1)` (choosing a random base between 256 and 2⁴⁰ when none is given) with `get(i, j)` for the hash of `s[i:j]` in O(1), and `equal(i, j, length)`: whether `s[i:i + length]` and `s[j:j + length]` are equal, by comparing hashes. Then `same_substrings(s, length)`: a dict from each **distinct** substring of that length to the sorted list of its start positions, grouping positions by hash and slicing the string only once per group.

In [ ]:
import random

class PrefixHash:
    def __init__(self, s, base=None, mod=(1 << 61) - 1):
        ___

def same_substrings(s, length):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_hash():\n    \"substring hashes\"\n    ph = PrefixHash(\"abracadabra\", base=131)\n    assert ph.get(0, 4) == ph.get(7, 11) and ph.get(0, 4) != ph.get(1, 5) and ph.get(3, 3) == 0\n    assert ph.equal(0, 7, 4) and not ph.equal(0, 1, 4) and ph.equal(2, 5, 0)\n    a, b = PrefixHash(\"xyz\"), PrefixHash(\"xyz\")\n    rng = random.Random(1)\n    s = \"\".join(rng.choice(\"ab\") for _ in range(200))\n    ph = PrefixHash(s)\n    for _ in range(2000):\n        i, j = rng.randrange(200), rng.randrange(200)\n        L = rng.randrange(0, 200 - max(i, j) + 1)\n        assert ph.equal(i, j, L) == (s[i:i + L] == s[j:j + L])\ndef test_groups():\n    \"grouping\"\n    assert same_substrings(\"abracadabra\", 4) == {\"abra\": [0, 7], \"brac\": [1], \"raca\": [2], \"acad\": [3], \"cada\": [4], \"adab\": [5], \"dabr\": [6]}\n    assert same_substrings(\"aaaa\", 2) == {\"aa\": [0, 1, 2]} and same_substrings(\"ab\", 3) == {}")

### Exercise: Distinct substrings of a length

Write `count_distinct(s, k)`: the number of distinct substrings of length k, using a set of rolling-hash values (not of strings). Then `kmer_similarity(a, b, k)`: the Jaccard similarity of the two strings' sets of length-k substrings (size of the intersection divided by size of the union, as a float; 1.0 if both sets are empty). Use one base for both strings.

In [ ]:
import random

def count_distinct(s, k):
    ___

def kmer_similarity(a, b, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_distinct():\n    \"distinct substrings\"\n    assert count_distinct(\"abracadabra\", 4) == 7 and count_distinct(\"aaaa\", 2) == 1 and count_distinct(\"ab\", 3) == 0\n    rng = random.Random(2)\n    for _ in range(100):\n        s = \"\".join(rng.choice(\"acgt\") for _ in range(rng.randint(0, 40)))\n        k = rng.randint(1, 6)\n        assert count_distinct(s, k) == len({s[i:i + k] for i in range(len(s) - k + 1)})\n    assert kmer_similarity(\"abcd\", \"bcde\", 2) == 2 / 4 and kmer_similarity(\"a\", \"b\", 2) == 1.0 and kmer_similarity(\"aaa\", \"aaa\", 1) == 1.0\n    body = source.split(\"def count_distinct\")[1]\n    assert \"s[i:i + k]\" not in body and \"[i:i+k]\" not in body")

### Exercise: The longest repeat

Write `longest_repeat(s)`: the longest substring occurring at least twice in s (overlaps allowed), choosing the one that occurs **first** if several have that length, and `""` if no character repeats. Use prefix hashes and binary search on the length. The checks run a 20,000-character string, where trying every length would be too slow.

In [ ]:
import random

def longest_repeat(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef slow(s):\n    for L in range(len(s) - 1, 0, -1):\n        starts = {}\n        for i in range(len(s) - L + 1):\n            starts.setdefault(s[i:i + L], []).append(i)\n        reps = [(v[0], k) for k, v in starts.items() if len(v) > 1]\n        if reps:\n            return min(reps)[1]\n    return \"\"\ndef test_repeat():\n    \"repeats\"\n    assert longest_repeat(\"banana\") == \"ana\" and longest_repeat(\"abcdefg\") == \"\" and longest_repeat(\"\") == \"\"\n    assert longest_repeat(\"aaaa\") == \"aaa\" and longest_repeat(\"to be or not to be\") == \"to be\"\n    rng = random.Random(3)\n    for _ in range(150):\n        s = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 25)))\n        assert longest_repeat(s) == slow(s), s\ndef test_big():\n    \"fast\"\n    rng = random.Random(4)\n    s = \"\".join(rng.choice(\"acgt\") for _ in range(20_000))\n    s = s[:12_000] + s[3_000:3_400] + s[12_000:]\n    t0 = time.perf_counter()\n    r = longest_repeat(s)\n    assert len(r) >= 400 and s.count(r) >= 1 and time.perf_counter() - t0 < 6")

### Exercise: The longest common substring

Write `longest_common(a, b)`: the longest string that is a substring of both, choosing the one that starts earliest **in a** on a tie (`""` if none). Hash both strings with the same base, binary search the length, and for each length put a's window hashes in a dict (hash to first position in a), then look up b's. Check the characters of the final answer, so a collision can never produce a wrong result.

In [ ]:
import random

def longest_common(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef slow(a, b):\n    for L in range(min(len(a), len(b)), 0, -1):\n        subs = {b[j:j + L] for j in range(len(b) - L + 1)}\n        for i in range(len(a) - L + 1):\n            if a[i:i + L] in subs:\n                return a[i:i + L]\n    return \"\"\ndef test_common():\n    \"common substrings\"\n    assert longest_common(\"xabcdey\", \"zzbcdq\") == \"bcd\" and longest_common(\"abc\", \"xyz\") == \"\" and longest_common(\"\", \"a\") == \"\"\n    assert longest_common(\"ab\", \"ba\") == \"a\"\n    rng = random.Random(5)\n    for _ in range(200):\n        a = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 15)))\n        b = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 15)))\n        assert longest_common(a, b) == slow(a, b), (a, b)\ndef test_big():\n    \"fast\"\n    rng = random.Random(6)\n    shared = \"\".join(rng.choice(\"acgt\") for _ in range(500))\n    a = \"\".join(rng.choice(\"acgt\") for _ in range(10_000)) + shared\n    b = shared + \"\".join(rng.choice(\"acgt\") for _ in range(10_000))\n    t0 = time.perf_counter()\n    assert len(longest_common(a, b)) >= 500 and time.perf_counter() - t0 < 6")

### Exercise: Comparing suffixes

Write `lcp(ph, i, j, n)`: the length of the longest common prefix of `s[i:]` and `s[j:]` (n is `len(s)`), by binary search with a `PrefixHash` (provided). Then `suffix_less(s, ph, i, j)`: whether `s[i:]` comes before `s[j:]` alphabetically, using `lcp` and one character comparison. Finally `suffix_array(s)`: the start positions of all suffixes in alphabetical order, sorting with `functools.cmp_to_key` (Week 19) and your comparison, without ever slicing the suffixes.

In [ ]:
import random
from functools import cmp_to_key

class PrefixHash:
    def __init__(self, s, base=None, mod=(1 << 61) - 1):
        base = base or random.randrange(256, 1 << 40)
        self.mod, self.h, self.pw = mod, [0] * (len(s) + 1), [1] * (len(s) + 1)
        for i, ch in enumerate(s):
            self.h[i + 1] = (self.h[i] * base + ord(ch)) % mod
            self.pw[i + 1] = self.pw[i] * base % mod

    def get(self, i, j):
        return (self.h[j] - self.h[i] * self.pw[j - i]) % self.mod

def lcp(ph, i, j, n):
    ___

def suffix_less(s, ph, i, j):
    ___

def suffix_array(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_suffixes():\n    \"suffix order\"\n    s = \"banana\"\n    ph = PrefixHash(s)\n    assert lcp(ph, 1, 3, 6) == 3 and lcp(ph, 0, 1, 6) == 0 and lcp(ph, 5, 5, 6) == 1\n    assert suffix_less(s, ph, 5, 3) and not suffix_less(s, ph, 0, 1)\n    assert suffix_array(\"banana\") == [5, 3, 1, 0, 4, 2] and suffix_array(\"\") == []\n    rng = random.Random(7)\n    for _ in range(100):\n        s = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 40)))\n        assert suffix_array(s) == sorted(range(len(s)), key=lambda i: s[i:])")

### Bug hunt: The wrong power

This substring hash shifts `h[i]` by `B ** j` instead of `B ** (j - i)`, so equal substrings at different places get different hashes: this prints `False`. Shift by the length of the substring so it prints `True`.

In [ ]:
B, M = 131, (1 << 61) - 1
s = "abcXabc"
h = [0]
for ch in s:
    h.append((h[-1] * B + ord(ch)) % M)

def sub(i, j):
    return (h[j] - h[i] * pow(B, j, M)) % M

print(sub(0, 3) == sub(4, 7))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"shift by the length\"\n    expect_output(\"True\")\n    assert sub(0, 3) != sub(1, 4)")

### Bug hunt: Trusting a tiny hash

This counts distinct substrings by their hashes, but with a modulus of 7 many different substrings collide, so the count is far too low: this prints `7`. Use the modulus 2⁶¹ − 1, so it prints `21`.

In [ ]:
B, M = 131, 7
s = "abcdefghijklmnopqrstuvwx"

def count_distinct(s, k):
    seen = set()
    for i in range(len(s) - k + 1):
        v = 0
        for ch in s[i:i + k]:
            v = (v * B + ord(ch)) % M
        seen.add(v)
    return len(seen)

print(count_distinct(s, 4))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a big modulus\"\n    expect_output(\"21\")")

### Bug hunt: Searching the wrong way

This binary search for the longest repeated length moves `hi` down when a repeat **is** found and `lo` up when it isn't, the wrong way round, so it ends on a length that doesn't repeat: this prints `''`. Swap the moves so it prints `'ana'`.

In [ ]:
def repeats(s, L):
    seen = set()
    for i in range(len(s) - L + 1):
        if s[i:i + L] in seen:
            return s[i:i + L]
        seen.add(s[i:i + L])
    return None

def longest_repeat(s):
    lo, hi, best = 1, len(s) - 1, ""
    while lo <= hi:
        mid = (lo + hi) // 2
        r = repeats(s, mid)
        if r is None:
            lo = mid + 1
        else:
            best, hi = r, mid - 1
    return best if repeats(s, len(best) + 1) is None else ""

print(repr(longest_repeat("banana")))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"longer when it works\"\n    expect_output(\"'ana'\")")

### Puzzle: Building prefix hashes

Arrange the lines to build prefix hashes and powers, and get a substring's hash. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Base 10, a large modulus, the digit string "27182818", with h[i] the hash of s[:i].

p = ___   # h[3]
q = ___   # the hash of s[2:5]
r = ___   # the hash of s[5:8]
s = ___   # the longest substring of "27182818" that occurs twice
t = ___   # the number of distinct substrings of length 2 in "27182818"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 271 and q == 182 and r == 818, \"p, q and r\"\n    assert s == \"18\" and t == 6, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: fingerprints for copied text

To spot copied passages between documents, comparing every substring is hopeless, but comparing **fingerprints** works. **Winnowing** (Saul Schleimer, Daniel Wilkerson and Alex Aiken, 2003, used in the MOSS system for detecting copied code) picks a small, robust set of hashes from each document:

1. Normalise the text: keep only letters and digits, lower-cased.
2. Hash every k-gram (substring of length k) with a rolling hash: a list of n − k + 1 hashes.
3. Slide a window of w consecutive hashes along that list; in each window, choose the **minimum** hash (the **rightmost** one if it occurs more than once). The fingerprints are the chosen `(hash, position)` pairs, each recorded once even when several windows choose it.

Any shared passage of at least w + k − 1 characters is guaranteed to produce a shared fingerprint. Write `fingerprints(text, k, w, base)`, returning a list of `(hash, position)` pairs in position order, finding each window's minimum with a **monotonic deque** (Week 43), so the whole thing is O(n). Then `similarity(a, b, k, w)`: the Jaccard similarity of the two documents' sets of fingerprint **hashes** (1.0 if both are empty), using one random base for both. Use the modulus 2⁶¹ − 1. The checks compare with a slow version and time a 100,000-character document.

In [ ]:
import random
from collections import deque

MOD = (1 << 61) - 1

def fingerprints(text, k, w, base):
    ___

def similarity(a, b, k, w):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef slow(text, k, w, base):\n    t = \"\".join(c.lower() for c in text if c.isalnum())\n    hs = []\n    for i in range(len(t) - k + 1):\n        v = 0\n        for ch in t[i:i + k]:\n            v = (v * base + ord(ch)) % MOD\n        hs.append(v)\n    chosen = []\n    for s in range(len(hs) - w + 1):\n        window = hs[s:s + w]\n        m = min(window)\n        pos = s + max(i for i, v in enumerate(window) if v == m)\n        if not chosen or chosen[-1][1] != pos:\n            chosen.append((m, pos))\n    out, seen = [], set()\n    for c in chosen:\n        if c not in seen:\n            seen.add(c)\n            out.append(c)\n    return sorted(out, key=lambda c: c[1])\ndef test_fingerprints():\n    \"against a slow version\"\n    rng = random.Random(8)\n    for _ in range(100):\n        text = \"\".join(rng.choice(\"ab c\") for _ in range(rng.randint(0, 60)))\n        k, w, base = rng.randint(1, 4), rng.randint(1, 5), rng.choice([3, 131, 257])\n        assert fingerprints(text, k, w, base) == slow(text, k, w, base), (text, k, w, base)\n    assert fingerprints(\"A do run run run, a do run run\", 5, 4, 131) == slow(\"A do run run run, a do run run\", 5, 4, 131)\ndef test_similarity():\n    \"similar documents\"\n    rng = random.Random(9)\n    words = [\"apple\", \"river\", \"stone\", \"cloud\", \"music\", \"bread\", \"light\", \"table\", \"green\", \"storm\"]\n    doc = \" \".join(rng.choice(words) for _ in range(300))\n    other = \" \".join(rng.choice(words) for _ in range(300))\n    copied = other[:600] + doc[500:1200] + other[600:]\n    assert similarity(doc, doc, 5, 4) == 1.0 and similarity(\"\", \"\", 5, 4) == 1.0\n    assert similarity(doc, copied, 8, 6) > similarity(doc, other, 8, 6)\ndef test_speed():\n    \"linear\"\n    rng = random.Random(10)\n    text = \"\".join(rng.choice(\"abcdefgh \") for _ in range(100_000))\n    t0 = time.perf_counter()\n    fingerprints(text, 8, 20, 131)\n    assert time.perf_counter() - t0 < 4")

## Warm-up

With base 10 and digits, a hash is just the number the digits spell, which makes it easy to see what's going on. `h[4]` is 3141 and `h[1]` is 3; shifting 3 left by three places gives 3000, and 3141 − 3000 = 141, which is exactly `s[1:4]`. Any substring, one subtraction.

## Prefix hashes

Week 44's polynomial hash of a string treats it as a number in base B, reduced modulo M:

hash(c₀c₁...cₖ₋₁) = c₀·B^(k−1) + c₁·B^(k−2) + ... + cₖ₋₁ (mod M)

Compute the hashes of **every prefix** once: `h[0] = 0` and `h[i + 1] = h[i]·B + code(s[i])`. Then, as in the warm-up, the hash of `s[i:j]` is

**hash(s[i:j]) = h[j] − h[i]·B^(j−i) (mod M)**

because `h[j]` is the hash of `s[:j]`, and `h[i]·B^(j−i)` is the hash of `s[:i]` shifted left past the j − i characters that follow it. Precompute the powers of B too, and every substring hash is O(1) after O(n) set-up. It's the same idea as Week 41's prefix sums, with multiplication by B playing the part of position.

## Collisions

Two different substrings can have the same hash: there are far more strings than values below M. If your code treats equal hashes as equal strings (and these algorithms must, to stay fast), a collision is a wrong answer. Two things keep the chance tiny:

- **A big prime modulus.** For two fixed different strings of length L and a **random** base, the chance their hashes collide is at most about L / M. With M = 2⁶¹ − 1 (a prime, and conveniently Python integers are unlimited, so no overflow worries), that's around 10⁻¹⁴ per comparison for strings of a thousand characters: even a billion comparisons give a collision chance near 10⁻⁵.
- **A random base**, chosen when the program starts. With a fixed, public base and modulus, someone who knows them can construct two different strings with the same hash on purpose (and for small moduli, it's easy: the birthday problem from Week 44 finds a pair among about √M random strings). A random base makes that impossible to plan in advance, the same reason Python randomises `hash()` for strings each run.

Some programs use **two** independent hashes (different moduli or bases) and compare both: a pair of 31-bit hashes behaves like one 62-bit hash. With Python's big integers, one hash modulo 2⁶¹ − 1 is simpler.

And when an occasional wrong answer is unacceptable, **check**: when two hashes match, compare the actual characters, as Rabin and Karp did. A false match costs one O(L) comparison, which is rare enough not to matter.

About a thousand strings were enough with M near a million (√M), exactly as the birthday problem predicts. With M = 2⁶¹ − 1 you'd need around 1.5 billion.

## Binary search on the answer

What's the **longest substring that occurs at least twice** in s? For a fixed length L, it's easy with hashes: put the hash of every length-L substring in a set, and see whether one repeats. That's O(n) per L. Trying every L would be O(n²), but notice the property is **monotonic**: if some substring of length L repeats, so does its first L − 1 characters. So the lengths that work form a block 0, 1, ..., L*, and binary search (Week 18) finds the largest in O(log n) tries: **O(n log n)** in all.

"Binary search on the answer" is a general technique: whenever a yes/no question about a number is monotonic (if it's true for L, it's true for everything smaller), binary search the largest L for which it's true.

## Two strings at once

The same machinery finds the **longest common substring** of two strings a and b: for a length L, hash every length-L substring of a into a set, then check b's against it. Monotonic again, so binary search L. The two strings need hashes with the **same base and modulus**, or equal substrings won't have equal hashes.

Hashes of fixed-length windows also **count distinct substrings** of length k: put them all in a set and take its size, O(n) rather than O(n·k) for a set of the strings themselves. In genetics, the distinct k-length substrings of a DNA sequence (its "k-mers") are a standard way to compare and index genomes.

## Ordering substrings

Hashes tell you whether two substrings are **equal**, but not which comes first alphabetically. They can still help: binary search the **longest common prefix** (LCP) of the two substrings, using hashes to test "are the first L characters equal?" (monotonic again). Then the first differing character decides the order. That's O(log n) per comparison, so sorting all n suffixes of a string this way takes O(n log² n), the basis of simple **suffix array** constructions used in text indexing.

## Practice

## Quiz

## Daily challenge

## Before you go

Take two paragraphs, one copied from the other with a few words changed, and work out by hand which 5-letter chunks they share. Tomorrow: palindromes, from checking one to finding the longest inside a string in linear time.